In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import gpboost as gpb

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
sns.set_theme(style="whitegrid")


c:\Users\ryanc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -----------------------------------------------------------------------------
# Config
# -----------------------------------------------------------------------------

OUTPUT_DIR = Path.cwd() / "notebook_exports"
ROUND_DATA_PATH = OUTPUT_DIR / "round_level_modeling_dataset_v1.parquet"

TEST_SIZE = 0.20
VALID_SIZE_WITHIN_TRAIN = 0.20
RANDOM_STATE = 42
TARGET_COL = "actual_round_strokes"

MPH_TO_MPS = 0.44704
MPS_TO_MPH = 2.23694

REFERENCE_WIND_MPH = 1.0
REFERENCE_WIND_MPS = REFERENCE_WIND_MPH * MPH_TO_MPS

REFERENCE_TEMP_F = 80.0
REFERENCE_TEMP_C = (REFERENCE_TEMP_F - 32.0) * 5.0 / 9.0

GPBOOST_PARAMS = {
    "boosting_type": "gbdt",
    "objective": "regression_l2",
    "n_estimators": 1000,
    "learning_rate": 0.02,
    "max_depth": 3,
    "num_leaves": 8,
    "min_child_samples": 100,
    "colsample_bytree": 0.7,
    "reg_lambda": 20.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "subsample": 1.0,
    "subsample_freq": 0,
    "verbose": 1,
}

print("Using dataset:", ROUND_DATA_PATH)
print("Reference wind mph:", REFERENCE_WIND_MPH)
print("Reference temp F:", REFERENCE_TEMP_F)

Using dataset: c:\Users\ryanc\dg_wind_effects\wind_impact_analysis\notebook_exports\round_level_modeling_dataset_v1.parquet
Reference wind mph: 1.0
Reference temp F: 80.0


In [3]:
# -----------------------------------------------------------------------------
# Load data
# -----------------------------------------------------------------------------

round_level_df = pd.read_parquet(ROUND_DATA_PATH).copy()

print("Round-level shape:", round_level_df.shape)
round_level_df =round_level_df[round_level_df.hole_count >= 18]
round_level_df.head()


Round-level shape: (419842, 28)


,event_year,tourn_id,round_number,player_key,course_id,layout_id,division,player_rating,actual_round_strokes,hole_count,round_total_hole_length,round_avg_hole_length,round_total_par,round_avg_hole_par,round_wind_speed_mps_mean,round_wind_speed_mps_max,round_wind_gust_mps_mean,round_wind_gust_mps_max,round_temp_c_mean,round_precip_mm_sum,round_precip_mm_mean,round_pressure_hpa_mean,round_humidity_pct_mean,round_strokes_over_par,round_length_over_par,wind_x_round_length,gust_x_round_length,wind_x_player_rating
0,2025,90004,1,PDGA#240866,248066,731340,MA4,777.0,60.0,18,4262.0,236.777778,54.0,3.000000,2.99,2.99,5.3,5.3,15.6,0.0,0.0,1018.5,82.0,6.0,78.925926,12743.38,22588.6,2323.23
1,2025,90022,1,PDGA#139372,303386,708703,MPO,944.0,65.0,22,6640.0,301.818182,69.0,3.136364,0.69,0.69,3.0,3.0,3.7,0.0,0.0,1009.0,83.0,-4.0,96.231884,4581.60,19920.0,651.36
2,2025,90022,1,PDGA#178707,303386,708703,FPO,921.0,67.0,22,6640.0,301.818182,69.0,3.136364,1.38,1.38,2.7,2.7,1.1,0.0,0.0,1009.0,91.0,-2.0,96.231884,9163.20,17928.0,1270.98
3,2025,90022,1,PDGA#235527,303386,708703,MA2,861.0,72.0,22,6640.0,301.818182,69.0,3.136364,1.25,1.25,2.3,2.3,2.2,0.0,0.0,1009.7,89.0,3.0,96.231884,8300.00,15272.0,1076.25
4,2025,90022,2,PDGA#208891,303386,708709,MA50,887.0,79.0,22,7337.0,333.500000,70.0,3.181818,1.71,1.71,4.2,4.2,5.5,0.0,0.0,1004.7,77.0,9.0,104.814286,12546.27,30815.4,1516.77


In [6]:
# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def regression_metrics(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)),
    }


def extract_gpboost_prediction_vector(pred_obj) -> np.ndarray:
    if isinstance(pred_obj, dict):
        if "response_mean" in pred_obj:
            return np.asarray(pred_obj["response_mean"]).reshape(-1)
        if "mu" in pred_obj:
            return np.asarray(pred_obj["mu"]).reshape(-1)
        if "fixed_effect" in pred_obj:
            return np.asarray(pred_obj["fixed_effect"]).reshape(-1)
        raise ValueError(f"Unsupported GPBoost prediction dict keys: {list(pred_obj.keys())}")

    arr = np.asarray(pred_obj)
    if arr.ndim == 0:
        return arr.reshape(1)
    return arr.reshape(-1)


def analysis_wind_bucket_from_mph(speed_mph: float | None) -> str:
    if speed_mph is None or pd.isna(speed_mph):
        return "unknown"
    if speed_mph < 3:
        return "0-3"
    if speed_mph < 6:
        return "3-6"
    if speed_mph < 9:
        return "6-9"
    if speed_mph < 12:
        return "9-12"
    if speed_mph < 15:
        return "12-15"
    return "15+"


def analysis_temp_bucket_from_f(temp_f: float | None) -> str:
    if temp_f is None or pd.isna(temp_f):
        return "unknown"
    if temp_f < 30:
        return "<30"
    if temp_f < 40:
        return "30-40"
    if temp_f < 50:
        return "40-50"
    if temp_f < 60:
        return "50-60"
    if temp_f < 70:
        return "60-70"
    if temp_f < 80:
        return "70-80"
    return "80+"


OBSERVED_WIND_BUCKETS = ["0-3", "3-6", "6-9", "9-12", "12-15", "15+"]
OBSERVED_TEMP_BUCKETS = ["<30", "30-40", "40-50", "50-60", "60-70", "70-80", "80+"]


def prepare_model_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["round_wind_speed_mps_mean"] = pd.to_numeric(out["round_wind_speed_mps_mean"], errors="coerce")
    out["round_temp_c_mean"] = pd.to_numeric(out["round_temp_c_mean"], errors="coerce")

    out["round_wind_speed_mph_mean"] = out["round_wind_speed_mps_mean"] * MPS_TO_MPH
    out["round_temp_f_mean"] = (pd.to_numeric(out["round_temp_c_mean"], errors="coerce") * 9.0 / 5.0) + 32.0

    out["observed_wind_bucket"] = out["round_wind_speed_mph_mean"].map(analysis_wind_bucket_from_mph)
    out["observed_temp_bucket"] = out["round_temp_f_mean"].map(analysis_temp_bucket_from_f)

    required_numeric = [
        TARGET_COL,
        "player_rating",
        "hole_count",
        "round_total_hole_length",
        "round_avg_hole_length",
        "round_total_par",
        "round_avg_hole_par",
        "round_length_over_par",
        "round_wind_speed_mps_mean",
        "round_temp_c_mean",
    ]

    for col in required_numeric:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=required_numeric + ["course_id", "division"]).copy()

    out["division"] = out["division"].astype("category")

    return out


def build_group_data_pred(df: pd.DataFrame) -> pd.DataFrame:
    return df[["course_id"]].copy()


def build_group_rand_coef_pred(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "wind_speed_mps_centered": df["round_wind_speed_mps_mean"] - REFERENCE_WIND_MPS,
            "temp_c_centered": df["round_temp_c_mean"] - REFERENCE_TEMP_C,
        }
    )


def build_weather_scenario_df(
    base_df: pd.DataFrame,
    *,
    wind_speed_mph: float | None = None,
    temperature_c: float | None = None,
) -> pd.DataFrame:
    out = base_df.copy()

    if wind_speed_mph is not None:
        out["round_wind_speed_mps_mean"] = wind_speed_mph * MPH_TO_MPS
        out["round_wind_speed_mph_mean"] = wind_speed_mph
        out["observed_wind_bucket"] = analysis_wind_bucket_from_mph(wind_speed_mph)

    if temperature_c is not None:
        out["round_temp_c_mean"] = float(temperature_c)
        out["round_temp_f_mean"] = (float(temperature_c) * 9.0 / 5.0) + 32.0
        out["observed_temp_bucket"] = analysis_temp_bucket_from_f(out["round_temp_f_mean"].iloc[0])

    out["division"] = out["division"].astype("category")

    return out


In [7]:
# -----------------------------------------------------------------------------
# Prepare modeling frame
# -----------------------------------------------------------------------------

model_df = prepare_model_frame(round_level_df)

NUMERIC_FIXED_FEATURES = [
    "player_rating",
    "hole_count",
    "round_total_hole_length",
    "round_avg_hole_length",
    "round_total_par",
    "round_avg_hole_par",
    "round_length_over_par",
    "round_wind_speed_mps_mean",
    "round_temp_c_mean",
]

CATEGORICAL_FIXED_FEATURES = [
    "division",
]

FIXED_FEATURE_COLS = NUMERIC_FIXED_FEATURES + CATEGORICAL_FIXED_FEATURES
RANDOM_EFFECT_GROUP_COL = "course_id"

print("Prepared rows:", len(model_df))
print("Course count:", model_df["course_id"].nunique())
print("Feature count:", len(FIXED_FEATURE_COLS))

model_df.head()


Prepared rows: 412671
Course count: 2979
Feature count: 10


,event_year,tourn_id,round_number,player_key,course_id,layout_id,division,player_rating,actual_round_strokes,hole_count,round_total_hole_length,round_avg_hole_length,round_total_par,round_avg_hole_par,round_wind_speed_mps_mean,round_wind_speed_mps_max,round_wind_gust_mps_mean,round_wind_gust_mps_max,round_temp_c_mean,round_precip_mm_sum,round_precip_mm_mean,round_pressure_hpa_mean,round_humidity_pct_mean,round_strokes_over_par,round_length_over_par,wind_x_round_length,gust_x_round_length,wind_x_player_rating,round_wind_speed_mph_mean,round_temp_f_mean,observed_wind_bucket,observed_temp_bucket
0,2025,90004,1,PDGA#240866,248066,731340,MA4,777.0,60.0,18,4262.0,236.777778,54.0,3.000000,2.99,2.99,5.3,5.3,15.6,0.0,0.0,1018.5,82.0,6.0,78.925926,12743.38,22588.6,2323.23,6.688451,60.08,6-9,60-70
1,2025,90022,1,PDGA#139372,303386,708703,MPO,944.0,65.0,22,6640.0,301.818182,69.0,3.136364,0.69,0.69,3.0,3.0,3.7,0.0,0.0,1009.0,83.0,-4.0,96.231884,4581.60,19920.0,651.36,1.543489,38.66,0-3,30-40
2,2025,90022,1,PDGA#178707,303386,708703,FPO,921.0,67.0,22,6640.0,301.818182,69.0,3.136364,1.38,1.38,2.7,2.7,1.1,0.0,0.0,1009.0,91.0,-2.0,96.231884,9163.20,17928.0,1270.98,3.086977,33.98,3-6,30-40
3,2025,90022,1,PDGA#235527,303386,708703,MA2,861.0,72.0,22,6640.0,301.818182,69.0,3.136364,1.25,1.25,2.3,2.3,2.2,0.0,0.0,1009.7,89.0,3.0,96.231884,8300.00,15272.0,1076.25,2.796175,35.96,0-3,30-40
4,2025,90022,2,PDGA#208891,303386,708709,MA50,887.0,79.0,22,7337.0,333.500000,70.0,3.181818,1.71,1.71,4.2,4.2,5.5,0.0,0.0,1004.7,77.0,9.0,104.814286,12546.27,30815.4,1516.77,3.825167,41.90,3-6,40-50


In [9]:
# -----------------------------------------------------------------------------
# Course support diagnostics for random slopes
# -----------------------------------------------------------------------------

course_support_df = (
    model_df.groupby("course_id", dropna=False)
    .agg(
        rows=("player_rating", "size"),
        layout_count=("layout_id", "nunique"),
        wind_speed_mph_min=("round_wind_speed_mph_mean", "min"),
        wind_speed_mph_max=("round_wind_speed_mph_mean", "max"),
        wind_speed_mph_std=("round_wind_speed_mph_mean", "std"),
        temp_f_min=("round_temp_f_mean", "min"),
        temp_f_max=("round_temp_f_mean", "max"),
        temp_f_std=("round_temp_f_mean", "std"),
    )
    .reset_index()
)

course_support_df["wind_speed_mph_range"] = (
    course_support_df["wind_speed_mph_max"] - course_support_df["wind_speed_mph_min"]
)
course_support_df["temp_f_range"] = (
    course_support_df["temp_f_max"] - course_support_df["temp_f_min"]
)

display(course_support_df.head(20))
display(course_support_df[["rows", "wind_speed_mph_range", "wind_speed_mph_std", "temp_f_range", "temp_f_std"]].describe())


,course_id,rows,layout_count,wind_speed_mph_min,wind_speed_mph_max,wind_speed_mph_std,temp_f_min,temp_f_max,temp_f_std,wind_speed_mph_range,temp_f_range
0,-1,24641,520,0.000000,21.228561,4.087342,-1.48,107.60,13.979336,21.228561,109.08
1,200380,435,51,1.454011,20.960128,3.919529,23.90,93.74,13.419400,19.506117,69.84
2,200382,31,2,0.357910,1.744813,0.481257,35.24,57.56,8.052181,1.386903,22.32
3,200383,249,13,1.409272,12.616342,2.253200,26.60,84.20,24.402864,11.207069,57.60
4,200385,243,2,1.118470,13.466379,4.217976,58.64,71.78,3.636345,12.347909,13.14
5,200387,121,2,3.019869,3.646212,0.259753,54.50,57.92,1.266447,0.626343,3.42
6,200390,88,2,1.498750,10.446510,2.439783,33.98,70.16,11.486562,8.947760,36.18
7,200392,246,6,0.156586,9.506995,2.215338,26.96,58.82,8.155625,9.350409,31.86
8,200393,37,2,3.668582,16.933636,4.487246,53.96,72.50,8.170840,13.265054,18.54
9,200394,72,10,0.559235,17.202069,3.771007,36.68,83.84,12.246186,16.642834,47.16


,rows,wind_speed_mph_range,wind_speed_mph_std,temp_f_range,temp_f_std
count,2979.000000,2979.000000,2978.000000,2979.000000,2978.000000
mean,138.526687,5.733212,1.813633,17.898671,5.766401
std,475.224672,4.775531,1.391262,17.568838,5.381873
min,1.000000,0.000000,0.000000,0.000000,0.000000
25%,38.000000,1.744813,0.684617,3.420000,1.368579
50%,84.000000,4.652835,1.587966,12.420000,4.180837
75%,162.000000,8.858282,2.674202,28.440000,8.862106
max,24641.000000,25.993243,8.632359,109.080000,36.198589


In [10]:
# -----------------------------------------------------------------------------
# Train / validation / test split
# -----------------------------------------------------------------------------

train_full_df, test_df = train_test_split(
    model_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

train_df, valid_df = train_test_split(
    train_full_df,
    test_size=VALID_SIZE_WITHIN_TRAIN,
    random_state=RANDOM_STATE,
)

print("Train rows:", len(train_df))
print("Valid rows:", len(valid_df))
print("Test rows:", len(test_df))


Train rows: 264108
Valid rows: 66028
Test rows: 82535


In [11]:
X_train = train_df[FIXED_FEATURE_COLS].copy()
y_train = train_df[TARGET_COL].copy()

X_valid = valid_df[FIXED_FEATURE_COLS].copy()
y_valid = valid_df[TARGET_COL].copy()

X_test = test_df[FIXED_FEATURE_COLS].copy()
y_test = test_df[TARGET_COL].copy()

group_train = build_group_data_pred(train_df)
group_valid = build_group_data_pred(valid_df)
group_test = build_group_data_pred(test_df)

group_rand_coef_train = build_group_rand_coef_pred(train_df)
group_rand_coef_valid = build_group_rand_coef_pred(valid_df)
group_rand_coef_test = build_group_rand_coef_pred(test_df)

gp_model = gpb.GPModel(
    group_data=group_train,
    group_rand_coef_data=group_rand_coef_train,
    ind_effect_group_rand_coef=[1, 1],
    likelihood="gaussian",
)

gpboost_model = gpb.GPBoostRegressor(**GPBOOST_PARAMS)

gpboost_model.fit(
    X_train,
    y_train,
    gp_model=gp_model,
    categorical_feature=CATEGORICAL_FIXED_FEATURES,
    eval_set=[(X_valid, y_valid)],
    use_gp_model_for_validation=False,
)


c:\Users\ryanc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\gpboost\basic.py:1858: UserWarning: Using categorical_feature in Dataset.
  _log_warning('Using categorical_feature in Dataset.')
c:\Users\ryanc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\gpboost\basic.py:1861: UserWarning: categorical_feature in Dataset is overridden.
New categorical_feature is ['division']
  _log_warning('categorical_feature in Dataset is overridden.\n'


[GPBoost] [Info] Total Bins 1786
[GPBoost] [Info] Number of data points in the train set: 264108, number of used features: 10
[GPBoost] [Info] [GPBoost with gaussian likelihood]: initscore=62.724972
[GPBoost] [Info] Start training from score 62.724972


c:\Users\ryanc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\gpboost\basic.py:1564: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning('Overriding the parameters from Reference Dataset.')
c:\Users\ryanc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\gpboost\basic.py:1375: UserWarning: categorical_column in param dict is overridden.
  _log_warning('{} in param dict is overridden.'.format(cat_alias))


[1]	valid_0's l2: 101.84
[2]	valid_0's l2: 101.824
[3]	valid_0's l2: 101.807
[4]	valid_0's l2: 101.79
[5]	valid_0's l2: 101.774
[6]	valid_0's l2: 101.763
[7]	valid_0's l2: 101.746
[8]	valid_0's l2: 101.735
[9]	valid_0's l2: 101.718
[10]	valid_0's l2: 101.711
[11]	valid_0's l2: 101.696
[12]	valid_0's l2: 101.68
[13]	valid_0's l2: 101.664
[14]	valid_0's l2: 101.648
[15]	valid_0's l2: 101.63
[16]	valid_0's l2: 101.613
[17]	valid_0's l2: 101.602
[18]	valid_0's l2: 101.584
[19]	valid_0's l2: 101.568
[20]	valid_0's l2: 101.55
[21]	valid_0's l2: 101.534
[22]	valid_0's l2: 101.523
[23]	valid_0's l2: 101.512
[24]	valid_0's l2: 101.494
[25]	valid_0's l2: 101.478
[26]	valid_0's l2: 101.46
[27]	valid_0's l2: 101.449
[28]	valid_0's l2: 101.431
[29]	valid_0's l2: 101.414
[30]	valid_0's l2: 101.397
[31]	valid_0's l2: 101.381
[32]	valid_0's l2: 101.373
[33]	valid_0's l2: 101.362
[34]	valid_0's l2: 101.355
[35]	valid_0's l2: 101.344
[36]	valid_0's l2: 101.328
[37]	valid_0's l2: 101.312
[38]	valid_0's l

,boosting_type,'gbdt'
,num_leaves,8
,max_depth,3
,learning_rate,0.02
,n_estimators,1000
,subsample_for_bin,200000
,objective,'regression_l2'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,100


In [ ]:
train_pred_raw = gpboost_model.predict(
    X_train,
    group_data_pred=group_train,
    group_rand_coef_data_pred=group_rand_coef_train,
    predict_response=True,
)

valid_pred_raw = gpboost_model.predict(
    X_valid,
    group_data_pred=group_valid,
    group_rand_coef_data_pred=group_rand_coef_valid,
    predict_response=True,
)

test_pred_raw = gpboost_model.predict(
    X_test,
    group_data_pred=group_test,
    group_rand_coef_data_pred=group_rand_coef_test,
    predict_response=True,
)

train_pred = extract_gpboost_prediction_vector(train_pred_raw)
valid_pred = extract_gpboost_prediction_vector(valid_pred_raw)
test_pred = extract_gpboost_prediction_vector(test_pred_raw)

metrics_df = pd.DataFrame(
    [
        {"split": "train", **regression_metrics(y_train, train_pred)},
        {"split": "valid", **regression_metrics(y_valid, valid_pred)},
        {"split": "test", **regression_metrics(y_test, test_pred)},
    ]
)

metrics_df


AttributeError: 'NoneType' object has no attribute 'predict'

In [ ]:
print("Estimated covariance parameters:")
cov_pars_df = gp_model.get_cov_pars(format_pandas=True)
cov_pars_df


In [ ]:
feature_importance_df = pd.DataFrame(
    {
        "feature": gpboost_model.feature_name_,
        "importance": gpboost_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

feature_importance_df


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=feature_importance_df,
    y="feature",
    x="importance",
    color="#4C78A8",
)
plt.title("GPBoost Fixed-Effect Feature Importances")
plt.xlabel("Importance")
plt.ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
wind_reference_df = build_weather_scenario_df(
    test_df,
    wind_speed_mph=REFERENCE_WIND_MPH,
)

temperature_reference_df = build_weather_scenario_df(
    test_df,
    temperature_c=REFERENCE_TEMP_C,
)

total_weather_reference_df = build_weather_scenario_df(
    test_df,
    wind_speed_mph=REFERENCE_WIND_MPH,
    temperature_c=REFERENCE_TEMP_C,
)

wind_reference_pred_raw = gpboost_model.predict(
    wind_reference_df[FIXED_FEATURE_COLS],
    group_data_pred=build_group_data_pred(wind_reference_df),
    group_rand_coef_data_pred=build_group_rand_coef_pred(wind_reference_df),
    predict_response=True,
)

temperature_reference_pred_raw = gpboost_model.predict(
    temperature_reference_df[FIXED_FEATURE_COLS],
    group_data_pred=build_group_data_pred(temperature_reference_df),
    group_rand_coef_data_pred=build_group_rand_coef_pred(temperature_reference_df),
    predict_response=True,
)

total_weather_reference_pred_raw = gpboost_model.predict(
    total_weather_reference_df[FIXED_FEATURE_COLS],
    group_data_pred=build_group_data_pred(total_weather_reference_df),
    group_rand_coef_data_pred=build_group_rand_coef_pred(total_weather_reference_df),
    predict_response=True,
)

wind_reference_pred = extract_gpboost_prediction_vector(wind_reference_pred_raw)
temperature_reference_pred = extract_gpboost_prediction_vector(temperature_reference_pred_raw)
total_weather_reference_pred = extract_gpboost_prediction_vector(total_weather_reference_pred_raw)

scored_test_df = test_df.copy()
scored_test_df["predicted_round_strokes"] = test_pred
scored_test_df["estimated_wind_effect_vs_reference"] = test_pred - wind_reference_pred
scored_test_df["estimated_temperature_effect_vs_reference"] = test_pred - temperature_reference_pred
scored_test_df["estimated_total_weather_effect_vs_reference"] = test_pred - total_weather_reference_pred

scored_test_df.head()


In [ ]:
effect_summary_df = pd.DataFrame(
    [
        {
            "metric": "wind",
            "mean": float(scored_test_df["estimated_wind_effect_vs_reference"].mean()),
            "median": float(scored_test_df["estimated_wind_effect_vs_reference"].median()),
            "std": float(scored_test_df["estimated_wind_effect_vs_reference"].std()),
            "p10": float(scored_test_df["estimated_wind_effect_vs_reference"].quantile(0.10)),
            "p90": float(scored_test_df["estimated_wind_effect_vs_reference"].quantile(0.90)),
        },
        {
            "metric": "temperature",
            "mean": float(scored_test_df["estimated_temperature_effect_vs_reference"].mean()),
            "median": float(scored_test_df["estimated_temperature_effect_vs_reference"].median()),
            "std": float(scored_test_df["estimated_temperature_effect_vs_reference"].std()),
            "p10": float(scored_test_df["estimated_temperature_effect_vs_reference"].quantile(0.10)),
            "p90": float(scored_test_df["estimated_temperature_effect_vs_reference"].quantile(0.90)),
        },
        {
            "metric": "total_weather",
            "mean": float(scored_test_df["estimated_total_weather_effect_vs_reference"].mean()),
            "median": float(scored_test_df["estimated_total_weather_effect_vs_reference"].median()),
            "std": float(scored_test_df["estimated_total_weather_effect_vs_reference"].std()),
            "p10": float(scored_test_df["estimated_total_weather_effect_vs_reference"].quantile(0.10)),
            "p90": float(scored_test_df["estimated_total_weather_effect_vs_reference"].quantile(0.90)),
        },
    ]
)

effect_summary_df


In [ ]:
effect_summary_df = pd.DataFrame(
    [
        {
            "metric": "wind",
            "mean": float(scored_test_df["estimated_wind_effect_vs_reference"].mean()),
            "median": float(scored_test_df["estimated_wind_effect_vs_reference"].median()),
            "std": float(scored_test_df["estimated_wind_effect_vs_reference"].std()),
            "p10": float(scored_test_df["estimated_wind_effect_vs_reference"].quantile(0.10)),
            "p90": float(scored_test_df["estimated_wind_effect_vs_reference"].quantile(0.90)),
        },
        {
            "metric": "temperature",
            "mean": float(scored_test_df["estimated_temperature_effect_vs_reference"].mean()),
            "median": float(scored_test_df["estimated_temperature_effect_vs_reference"].median()),
            "std": float(scored_test_df["estimated_temperature_effect_vs_reference"].std()),
            "p10": float(scored_test_df["estimated_temperature_effect_vs_reference"].quantile(0.10)),
            "p90": float(scored_test_df["estimated_temperature_effect_vs_reference"].quantile(0.90)),
        },
        {
            "metric": "total_weather",
            "mean": float(scored_test_df["estimated_total_weather_effect_vs_reference"].mean()),
            "median": float(scored_test_df["estimated_total_weather_effect_vs_reference"].median()),
            "std": float(scored_test_df["estimated_total_weather_effect_vs_reference"].std()),
            "p10": float(scored_test_df["estimated_total_weather_effect_vs_reference"].quantile(0.10)),
            "p90": float(scored_test_df["estimated_total_weather_effect_vs_reference"].quantile(0.90)),
        },
    ]
)

effect_summary_df


In [ ]:
def build_wind_curve_for_model(
    model,
    base_df: pd.DataFrame,
    *,
    reference_wind_mph: float,
    candidate_wind_mph_values: list[float],
) -> pd.DataFrame:
    baseline_df = build_weather_scenario_df(
        base_df,
        wind_speed_mph=reference_wind_mph,
    )
    baseline_pred_raw = model.predict(
        baseline_df[FIXED_FEATURE_COLS],
        group_data_pred=build_group_data_pred(baseline_df),
        group_rand_coef_data_pred=build_group_rand_coef_pred(baseline_df),
        predict_response=True,
    )
    baseline_pred = extract_gpboost_prediction_vector(baseline_pred_raw)

    rows = []

    for wind_mph in candidate_wind_mph_values:
        scenario_df = build_weather_scenario_df(
            base_df,
            wind_speed_mph=wind_mph,
        )
        scenario_pred_raw = model.predict(
            scenario_df[FIXED_FEATURE_COLS],
            group_data_pred=build_group_data_pred(scenario_df),
            group_rand_coef_data_pred=build_group_rand_coef_pred(scenario_df),
            predict_response=True,
        )
        scenario_pred = extract_gpboost_prediction_vector(scenario_pred_raw)

        effect_values = scenario_pred - baseline_pred

        rows.append(
            {
                "wind_speed_mph": wind_mph,
                "wind_bucket": analysis_wind_bucket_from_mph(wind_mph),
                "effect_mean": float(np.mean(effect_values)),
                "effect_p10": float(np.quantile(effect_values, 0.10)),
                "effect_p90": float(np.quantile(effect_values, 0.90)),
            }
        )

    return pd.DataFrame(rows)


def build_temperature_curve_for_model(
    model,
    base_df: pd.DataFrame,
    *,
    reference_temp_c: float,
    candidate_temp_c_values: list[float],
) -> pd.DataFrame:
    baseline_df = build_weather_scenario_df(
        base_df,
        temperature_c=reference_temp_c,
    )
    baseline_pred_raw = model.predict(
        baseline_df[FIXED_FEATURE_COLS],
        group_data_pred=build_group_data_pred(baseline_df),
        group_rand_coef_data_pred=build_group_rand_coef_pred(baseline_df),
        predict_response=True,
    )
    baseline_pred = extract_gpboost_prediction_vector(baseline_pred_raw)

    rows = []

    for temp_c in candidate_temp_c_values:
        scenario_df = build_weather_scenario_df(
            base_df,
            temperature_c=temp_c,
        )
        scenario_pred_raw = model.predict(
            scenario_df[FIXED_FEATURE_COLS],
            group_data_pred=build_group_data_pred(scenario_df),
            group_rand_coef_data_pred=build_group_rand_coef_pred(scenario_df),
            predict_response=True,
        )
        scenario_pred = extract_gpboost_prediction_vector(scenario_pred_raw)

        effect_values = scenario_pred - baseline_pred
        temp_f = (temp_c * 9.0 / 5.0) + 32.0

        rows.append(
            {
                "temperature_c": temp_c,
                "temperature_f": temp_f,
                "temp_bucket": analysis_temp_bucket_from_f(temp_f),
                "effect_mean": float(np.mean(effect_values)),
                "effect_p10": float(np.quantile(effect_values, 0.10)),
                "effect_p90": float(np.quantile(effect_values, 0.90)),
            }
        )

    return pd.DataFrame(rows)


In [ ]:
candidate_wind_mph_values = [1.5, 4.5, 7.5, 10.5, 13.5, 17.0]
candidate_temp_f_values = [25, 35, 45, 55, 65, 75, 85]
candidate_temp_c_values = [(f - 32.0) * 5.0 / 9.0 for f in candidate_temp_f_values]

wind_curve_df = build_wind_curve_for_model(
    gpboost_model,
    test_df,
    reference_wind_mph=REFERENCE_WIND_MPH,
    candidate_wind_mph_values=candidate_wind_mph_values,
)

temperature_curve_df = build_temperature_curve_for_model(
    gpboost_model,
    test_df,
    reference_temp_c=REFERENCE_TEMP_C,
    candidate_temp_c_values=candidate_temp_c_values,
)

display(wind_curve_df)
display(temperature_curve_df)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    wind_curve_df["wind_speed_mph"],
    wind_curve_df["effect_mean"],
    marker="o",
    linewidth=2,
    color="#2ca02c",
)
axes[0].fill_between(
    wind_curve_df["wind_speed_mph"],
    wind_curve_df["effect_p10"],
    wind_curve_df["effect_p90"],
    color="#2ca02c",
    alpha=0.15,
)
axes[0].axhline(0.0, color="black", linestyle="--", linewidth=1)
axes[0].axvline(REFERENCE_WIND_MPH, color="gray", linestyle=":", linewidth=1)
axes[0].set_title("Wind Speed Effect vs Reference")
axes[0].set_xlabel("Wind speed mean (mph)")
axes[0].set_ylabel("strokes")

axes[1].plot(
    temperature_curve_df["temperature_f"],
    temperature_curve_df["effect_mean"],
    marker="o",
    linewidth=2,
    color="#ff7f0e",
)
axes[1].fill_between(
    temperature_curve_df["temperature_f"],
    temperature_curve_df["effect_p10"],
    temperature_curve_df["effect_p90"],
    color="#ff7f0e",
    alpha=0.15,
)
axes[1].axhline(0.0, color="black", linestyle="--", linewidth=1)
axes[1].axvline(REFERENCE_TEMP_F, color="gray", linestyle=":", linewidth=1)
axes[1].set_title("Temperature Effect vs Reference")
axes[1].set_xlabel("Temperature (F)")
axes[1].set_ylabel("strokes")

plt.tight_layout()
plt.show()


In [ ]:
wind_effect_boxplot_df = scored_test_df.copy()

wind_effect_boxplot_df["observed_wind_bucket_str"] = (
    wind_effect_boxplot_df["observed_wind_bucket"]
    .astype(str)
    .str.strip()
)

wind_effect_boxplot_df["observed_wind_bucket_plot"] = pd.Categorical(
    wind_effect_boxplot_df["observed_wind_bucket_str"],
    categories=OBSERVED_WIND_BUCKETS,
    ordered=True,
)

wind_effect_bucket_summary_df = (
    wind_effect_boxplot_df.groupby("observed_wind_bucket_str", dropna=False)
    .agg(
        rows=("player_rating", "size"),
        avg_observed_wind_mph=("round_wind_speed_mph_mean", "mean"),
        mean_effect=("estimated_wind_effect_vs_reference", "mean"),
        median_effect=("estimated_wind_effect_vs_reference", "median"),
        p10_effect=("estimated_wind_effect_vs_reference", lambda s: s.quantile(0.10)),
        p25_effect=("estimated_wind_effect_vs_reference", lambda s: s.quantile(0.25)),
        p75_effect=("estimated_wind_effect_vs_reference", lambda s: s.quantile(0.75)),
        p90_effect=("estimated_wind_effect_vs_reference", lambda s: s.quantile(0.90)),
    )
    .reset_index()
)

wind_effect_bucket_summary_df["bucket_order"] = wind_effect_bucket_summary_df[
    "observed_wind_bucket_str"
].map({bucket: idx for idx, bucket in enumerate(OBSERVED_WIND_BUCKETS)})

wind_effect_bucket_summary_df = (
    wind_effect_bucket_summary_df.sort_values("bucket_order")
    .drop(columns="bucket_order")
    .reset_index(drop=True)
)

wind_effect_bucket_summary_df


In [ ]:
plt.figure(figsize=(11, 6))

sns.boxplot(
    data=wind_effect_boxplot_df,
    x="observed_wind_bucket_plot",
    y="estimated_wind_effect_vs_reference",
    showfliers=False,
    color="#7DBE7A",
)

plt.axhline(0.0, color="black", linestyle="--", linewidth=1)
plt.title("Estimated Wind Effect by Observed Wind-Speed Bucket")
plt.xlabel("Observed wind-speed bucket (mph)")
plt.ylabel("Estimated wind effect vs reference (strokes)")
plt.tight_layout()
plt.show()


In [ ]:
course_reference_df = build_weather_scenario_df(
    test_df,
    wind_speed_mph=REFERENCE_WIND_MPH,
    temperature_c=REFERENCE_TEMP_C,
)

course_wind_shift_df = build_weather_scenario_df(
    test_df,
    wind_speed_mph=10.5,
    temperature_c=REFERENCE_TEMP_C,
)

course_temp_shift_df = build_weather_scenario_df(
    test_df,
    wind_speed_mph=REFERENCE_WIND_MPH,
    temperature_c=(60.0 - 32.0) * 5.0 / 9.0,
)

course_reference_pred = extract_gpboost_prediction_vector(
    gpboost_model.predict(
        course_reference_df[FIXED_FEATURE_COLS],
        group_data_pred=build_group_data_pred(course_reference_df),
        group_rand_coef_data_pred=build_group_rand_coef_pred(course_reference_df),
        predict_response=True,
    )
)

course_wind_shift_pred = extract_gpboost_prediction_vector(
    gpboost_model.predict(
        course_wind_shift_df[FIXED_FEATURE_COLS],
        group_data_pred=build_group_data_pred(course_wind_shift_df),
        group_rand_coef_data_pred=build_group_rand_coef_pred(course_wind_shift_df),
        predict_response=True,
    )
)

course_temp_shift_pred = extract_gpboost_prediction_vector(
    gpboost_model.predict(
        course_temp_shift_df[FIXED_FEATURE_COLS],
        group_data_pred=build_group_data_pred(course_temp_shift_df),
        group_rand_coef_data_pred=build_group_rand_coef_pred(course_temp_shift_df),
        predict_response=True,
    )
)

course_response_diag_df = test_df[
    [
        "course_id",
        "round_wind_speed_mph_mean",
        "round_temp_f_mean",
    ]
].copy()

course_response_diag_df["wind_effect_10_5mph_vs_reference"] = course_wind_shift_pred - course_reference_pred
course_response_diag_df["temp_effect_60F_vs_reference"] = course_temp_shift_pred - course_reference_pred

course_response_summary_df = (
    course_response_diag_df.groupby("course_id", dropna=False)
    .agg(
        rows=("course_id", "size"),
        avg_observed_wind_mph=("round_wind_speed_mph_mean", "mean"),
        avg_observed_temp_f=("round_temp_f_mean", "mean"),
        mean_wind_effect_10_5mph=("wind_effect_10_5mph_vs_reference", "mean"),
        median_wind_effect_10_5mph=("wind_effect_10_5mph_vs_reference", "median"),
        mean_temp_effect_60F=("temp_effect_60F_vs_reference", "mean"),
        median_temp_effect_60F=("temp_effect_60F_vs_reference", "median"),
    )
    .reset_index()
)

display(course_response_summary_df.sort_values("mean_wind_effect_10_5mph", ascending=False).head(20))
display(course_response_summary_df.sort_values("mean_wind_effect_10_5mph", ascending=True).head(20))
display(course_response_summary_df.sort_values("mean_temp_effect_60F", ascending=False).head(20))
display(course_response_summary_df.sort_values("mean_temp_effect_60F", ascending=True).head(20))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    course_response_summary_df["mean_wind_effect_10_5mph"],
    bins=40,
    color="#2ca02c",
    ax=axes[0],
)
axes[0].axvline(0.0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Distribution of Course-Level Wind Response")
axes[0].set_xlabel("Mean course effect: 10.5 mph vs reference")

sns.histplot(
    course_response_summary_df["mean_temp_effect_60F"],
    bins=40,
    color="#ff7f0e",
    ax=axes[1],
)
axes[1].axvline(0.0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Distribution of Course-Level Temperature Response")
axes[1].set_xlabel("Mean course effect: 60F vs reference")

plt.tight_layout()
plt.show()
